# Calculadora para el parcial de LDBN
Sistemas clásicos, probabilísticos y cuánticos.

**Convención del curso:** `M[destino, origen]` → columna = de dónde sale, fila = a dónde llega.

**Cómo usarlo:** ejecuta primero la celda de *Funciones* (Shift+Enter). Luego, en cada sección, **cambia los datos de la celda y ejecútala**.

**Formato de datos:** matrices `"0,1;1,0"` (filas separadas por `;`), vectores `"1/sqrt(2), i/sqrt(2)"`. Acepta `i`, `sqrt()`, fracciones. Decimales con punto.

## 0. Funciones (ejecutar primero)

In [ ]:
import re
import numpy as np

np.set_printoptions(precision=4, suppress=True, linewidth=120)
TOL = 1e-9


# ----------------------------------------------------------------------------
# Lectura de datos
# ----------------------------------------------------------------------------
def _num(txt):
    txt = txt.strip().replace("^", "**")
    txt = re.sub(r"(?<![A-Za-z_])i(?![A-Za-z_(])", "j", txt)  # i -> j
    txt = re.sub(r"\)(?=[jA-Za-z0-9(])", ")*", txt)            # )j, )( -> )*j, )*(
    env = {"sqrt": np.sqrt, "pi": np.pi, "j": 1j, "exp": np.exp}
    return complex(eval(txt, {"__builtins__": {}}, env))


def vec(txt):
    """'1,2,3' -> vector columna (array 1D)."""
    return np.array([_num(t) for t in txt.split(",") if t.strip()], dtype=complex)


def mat(txt):
    """'0,1;1,0' -> matriz."""
    filas = [[_num(t) for t in f.split(",") if t.strip()] for f in txt.split(";") if f.strip()]
    return np.array(filas, dtype=complex)


def show(x):
    """Imprime quitando parte imaginaria si es ~0."""
    x = np.array(x)
    if np.all(np.abs(x.imag) < TOL):
        x = x.real
    print(x)


# ----------------------------------------------------------------------------
# Sistemas clásicos / grafos
# ----------------------------------------------------------------------------
def matriz_desde_flechas(n, flechas):
    """flechas = [(origen, destino), ...]  ->  M[destino, origen] = 1"""
    M = np.zeros((n, n))
    for o, d in flechas:
        M[d, o] = 1
    return M


def evolucionar(M, v0, clics):
    """Devuelve la lista [v0, v1, ..., v_clics] (cada clic: v <- M v)."""
    v = np.array(v0, dtype=complex)
    hist = [v]
    for _ in range(clics):
        v = M @ v
        hist.append(v)
    return hist


def potencia(M, n):
    return np.linalg.matrix_power(M, n)


def caminos(M, origen, destino, n):
    """Número de caminos de n pasos origen -> destino = (M^n)[destino, origen]."""
    return potencia(M, n)[destino, origen]


# ----------------------------------------------------------------------------
# Validación de estados y matrices
# ----------------------------------------------------------------------------
def es_estado_probabilistico(v):
    v = np.array(v)
    reales = np.all(np.abs(v.imag) < TOL)
    no_neg = np.all(v.real >= -TOL)
    suma = v.sum().real
    return bool(reales and no_neg and abs(suma - 1) < 1e-6), suma


def es_estado_cuantico(v):
    n2 = np.sum(np.abs(v) ** 2)
    return bool(abs(n2 - 1) < 1e-6), n2


def es_estocastica_por_columnas(M):
    M = np.array(M)
    ok = np.all(M.real >= -TOL) and np.allclose(M.sum(axis=0), 1)
    return bool(ok), M.sum(axis=0)


def es_hermitiana(M):
    M = np.array(M)
    return bool(np.allclose(M, M.conj().T))


def es_unitaria(M):
    M = np.array(M)
    return bool(np.allclose(M.conj().T @ M, np.eye(M.shape[0])))


# ----------------------------------------------------------------------------
# Cuántica: medición, valor esperado, varianza, vectores propios
# ----------------------------------------------------------------------------
def bra(v):
    return np.conj(v)


def prob_posicion(v):
    """Probabilidad de cada punto: |c_k|^2 (base estándar)."""
    return np.abs(np.array(v)) ** 2


def valor_esperado(Om, psi):
    return np.vdot(psi, Om @ psi)  # <psi|Om|psi>


def varianza(Om, psi):
    e = valor_esperado(Om, psi)
    e2 = valor_esperado(Om @ Om, psi)
    return (e2 - e ** 2).real


def valores_propios(Om):
    """Para hermitianas (eigh): valores reales ordenados y vectores propios."""
    return np.linalg.eigh(np.array(Om, dtype=complex))


def prob_medicion(Om, psi):
    """Posibles lecturas del observable y su probabilidad (agrupa degenerados)."""
    w, V = valores_propios(Om)
    psi = np.array(psi, dtype=complex)
    res = {}
    for k, lam in enumerate(w):
        p = abs(np.vdot(V[:, k], psi)) ** 2
        clave = round(float(lam), 6)
        res[clave] = res.get(clave, 0) + p
    return res


def es_estado_propio(Om, v, tol=1e-2):
    """Devuelve (¿propio?, lambda, residuo relativo)."""
    v = np.array(v, dtype=complex)
    w = Om @ v
    lam = np.vdot(v, w) / np.vdot(v, v)
    resid = np.linalg.norm(w - lam * v) / max(np.linalg.norm(w), 1e-15)
    return bool(resid < tol), lam, resid


def proyeccion_colapso(psi, base_vec):
    """Probabilidad y estado tras colapsar sobre el subespacio generado por base_vec."""
    b = np.array(base_vec, dtype=complex)
    b = b / np.linalg.norm(b)
    amp = np.vdot(b, psi)
    return abs(amp) ** 2, b


# ----------------------------------------------------------------------------
# Probabilidades en etapas (rendijas, etc.)
# ----------------------------------------------------------------------------
def etapas(*matrices, v0):
    """Aplica M1, luego M2, ... : resultado = Mn ... M2 M1 v0."""
    v = np.array(v0, dtype=float)
    for M in matrices:
        v = np.array(M) @ v
    return v

## 1. Grafo → matriz, y evolución de canicas (P3, P9, P15)
Flechas como `(origen, destino)`. Cambia `n`, `flechas`, `v0` y `clics`.

In [ ]:
n = 6
flechas = [(0,1),(1,2),(2,5),(5,1),(4,5),(3,3)]
v0 = [4,1,5,3,10,2]
clics = 2

M = matriz_desde_flechas(n, flechas)
print("M[destino, origen]:"); show(M)
print("\nSuma de columnas (1 = cada punto tiene una flecha saliente):", M.sum(axis=0))
for t, v in enumerate(evolucionar(M, v0, clics)):
    print(f"clic {t}:", v.real)

## 2. Potencias de M y número de caminos (P4)

In [ ]:
M = mat("0,1,0,0;0,0,1,0;0,0,0,1;1,0,0,0")
pasos = 2
origen, destino = 3, 1

Mn = potencia(M, pasos)
print(f"M^{pasos}:"); show(Mn)
print(f"Caminos de {pasos} pasos {origen} -> {destino}:", Mn[destino, origen].real)

## 3. ¿Es un estado válido? (P8)
Probabilístico: entradas reales ≥ 0 que suman 1. Cuántico: suma de |c|² = 1.

In [ ]:
candidatos = ["1/5, 2/3, 2/15", "sqrt(2), sqrt(2)", "1/sqrt(2), 1/sqrt(2)", "0, 1", "0.1, 0.9"]
for s in candidatos:
    v = vec(s)
    okp, suma = es_estado_probabilistico(v)
    okq, n2 = es_estado_cuantico(v)
    print(f"{s:28s} probabilístico: {okp!s:5}  (suma = {suma:.4f})   cuántico: {okq!s:5}  (Σ|c|² = {n2:.4f})")

## 4. Probabilidad de cada posición |c_k|² (P1)

In [ ]:
psi = vec("(sqrt(5)+2i)/sqrt(30), (-2-sqrt(5)i)/sqrt(30), i*sqrt(12)/sqrt(30)")
p = prob_posicion(psi)
for k, x in enumerate(p):
    print(f"punto {k}: {x:.4f} = {100*x:.2f} %")
print("suma:", p.sum())

## 5. Valor esperado y varianza (P5, P6)

In [ ]:
Om  = mat("0,-i;i,0")
psi = vec("1/sqrt(2), i/sqrt(2)")

print("¿Om hermitiana?:", es_hermitiana(Om))
print("Norma² de psi:", np.sum(np.abs(psi)**2).round(6))
print("<Om> =", np.round(valor_esperado(Om, psi), 6))
print("Var  =", round(varianza(Om, psi), 6))

## 6. Lecturas posibles y sus probabilidades (P2, P10, espín)
Las lecturas son los **valores propios**; la probabilidad de cada una es |⟨v_propio|ψ⟩|².

In [ ]:
Om  = mat("0,1;1,0")
psi = vec("1, 0")

w, V = valores_propios(Om)
print("Valores propios:", np.round(w, 6))
print("Vectores propios (columnas):"); show(V)
print()
for lam, p in prob_medicion(Om, psi).items():
    print(f"lectura {lam:+.4f}   prob = {p:.4f} ({100*p:.2f} %)")

## 7. ¿Cuál candidato es estado propio? (pregunta de H)
Muestra λ y el residuo. Residuo 0 = propio exacto; si ninguno lo es, el menor es el más probable.

In [ ]:
H = mat("1,0,0,0; 0,0,-i,0; 0,i,0,0; 0,0,0,1")
candidatos = {
    "a": "0, 1, 1, 0",
    "b": "1, 0, 0, 1",
    "c": "0, 1, 0, 0",
    "d": "0, 1, i, 0",
}
print("¿H hermitiana?:", es_hermitiana(H), "\n")
for nombre, s in candidatos.items():
    ok, lam, r = es_estado_propio(H, vec(s))
    print(f"{nombre}: lambda = {np.round(lam,4)}   residuo = {r:.4f}   {'PROPIO' if ok else 'no propio exacto'}")

## 8. Determinante, traza, hermitiana, unitaria, valores propios

In [ ]:
A = mat("1,1;1,-1") / np.sqrt(2)

print("Determinante:", np.round(np.linalg.det(A), 6))
print("Traza:", np.round(np.trace(A), 6))
print("Hermitiana:", es_hermitiana(A), "| Unitaria:", es_unitaria(A))
print("Valores propios:", np.round(np.linalg.eigvals(A), 6))

## 9. Dinámica cuántica: M·v (P7)
Una dinámica cuántica debe ser unitaria; así conserva la norma.

In [ ]:
M = mat("1/sqrt(2), 1/sqrt(2); 1/sqrt(2), -1/sqrt(2)")
v = vec("1, 0")

print("¿M unitaria?:", es_unitaria(M))
V = M @ v
print("V = M v =", np.round(V, 4))
print("Norma² de V:", np.round(np.sum(np.abs(V)**2), 6))
print("M V =", np.round(M @ V, 4), "  (¿estado básico?)")

## 10. Probabilidades en etapas (P12: rendijas)
Se multiplican las matrices de cada etapa: resultado = M₂ · M₁ · v₀.
Aquí: arco → 3 rendijas (1/3 cada una) → 11 blancos (10, 15, 50, 15, 10 %).

In [ ]:
n_blancos = 11
centros = [3, 6, 9]                    # blanco frente a cada rendija (numerados 1..11)
perfil  = [0.10, 0.15, 0.50, 0.15, 0.10]
blanco_A = 5

M1 = np.ones((3, 1)) / 3               # arco -> rendijas
M2 = np.zeros((n_blancos, 3))          # rendijas -> blancos
for k, c in enumerate(centros):
    for off, p in zip(range(-2, 3), perfil):
        M2[c - 1 + off, k] = p

res = etapas(M1, M2, v0=[1])
print("Prob. de cada blanco (%):", np.round(res * 100, 2))
print(f"Blanco A (#{blanco_A}): {res[blanco_A-1]*100:.2f} %  -> aproximado: {round(res[blanco_A-1]*100)}")